## Building GeoJSON files from our annotation volumes

In [1]:
import os
import json
import plotly.graph_objects as go

from plotlybrain.build_geoJSON import (
    load_annotation_volume,
    load_structure_graph,
    build_geojson,
    scale_cartesian_to_lonlat,
)

/home/ateruels@neuro.uni-bremen.de/PlotlyBrain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
volume, header = load_annotation_volume(resolution_um=25)
structure_df = load_structure_graph()

print("Volume shape:", volume.shape)
print("Number of ontology regions:", len(structure_df))


Volume shape: (528, 320, 456)
Number of ontology regions: 1327


In [3]:
# 2. Build one real slice
geojson = build_geojson(
    volume=volume,
    structure_df=structure_df,
    orientation="coronal",
    resolution_um=25,
    coords_mm=[-2.0, -3.0],
    min_area_px=5,
    simplify_px=0.8,
    polygon_mode="contour",
    smooth_sigma=1.0,
)

Building GeoJSON slices: 100%|██████████| 2/2 [00:00<00:00,  3.59it/s]


In [4]:
geojson_lonlat = scale_cartesian_to_lonlat(
    geojson,
    lon_range = (-15.0, 15.0),
    lat_range = (-10.0, 10.0),
    )

Testing some code to plot multiple slides at a time

In [6]:
fig = go.Figure()

coords = [-2.0, -3.0]

offset_by_coord = {
    coord: i * 35
    for i, coord in enumerate(coords)
}

for feature in geojson_lonlat["features"]:
    region_name = feature["properties"]["Region name"]
    region_id = feature["properties"]["Region ID"]
    coord_mm = feature["properties"]["coordinate_mm"]

    if region_id in [0, 997]:
        continue

    x_offset = offset_by_coord[coord_mm]

    for polygon in feature["geometry"]["coordinates"]:
        for ring in polygon:
            lon = [p[0] + x_offset for p in ring]
            lat = [p[1] for p in ring]

            fig.add_trace(
                go.Scattergeo(
                    lon=lon,
                    lat=lat,
                    mode="lines",
                    line=dict(color="black", width=0.5),
                    hovertemplate=(
                        f"<b>{region_name}</b><br>"
                        f"Region ID: {region_id}<br>"
                        f"AP: {coord_mm:.2f} mm"
                        "<extra></extra>"
                    ),
                    showlegend=False,
                )
            )

fig.update_geos(
    visible=False,
    fitbounds="locations",
    projection_type="equirectangular",
)

fig.update_layout(
    title="Allen CCF coronal slices",
    width=1000,
    height=500,
    margin=dict(l=0, r=0, t=40, b=0),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

fig.show()

Testing some code to write a slider version

In [7]:
import plotly.graph_objects as go

coords_mm = [-2.0, -3.0]

fig = go.Figure()
trace_groups = []
for coord in coords_mm:
    trace_indices = []
    features_this_slice = [
        f for f in geojson_lonlat["features"]
        if round(f["properties"]["coordinate_mm"], 3) == round(coord, 3)
    ]
    for feature in features_this_slice:
        region_name = feature["properties"]["Region name"]
        region_id = feature["properties"]["Region ID"]

        if region_id in [0, 997]:
            continue
        for polygon in feature["geometry"]["coordinates"]:
            for ring in polygon:
                lon = [p[0] for p in ring]
                lat = [p[1] for p in ring]

                fig.add_trace(
                    go.Scattergeo(
                        lon=lon,
                        lat=lat,
                        mode="lines",
                        line=dict(color="black", width=0.5),
                        hovertemplate=(
                            f"<b>{region_name}</b><br>"
                            f"Region ID: {region_id}<br>"
                            f"AP: {coord:.2f} mm"
                            "<extra></extra>"
                        ),
                        visible=(coord == coords_mm[0]),
                        showlegend=False,
                    )
                )

                trace_indices.append(len(fig.data) - 1)

    trace_groups.append(trace_indices)

steps = []

for i, coord in enumerate(coords_mm):
    visible = [False] * len(fig.data)

    for trace_idx in trace_groups[i]:
        visible[trace_idx] = True

    steps.append(
        dict(
            method="update",
            label=f"{coord:.2f} mm",
            args=[
                {"visible": visible},
                {"title": f"Allen CCF coronal slice, AP = {coord:.2f} mm"},
            ],
        )
    )

fig.update_layout(
    sliders=[
        dict(
            active=0,
            currentvalue={"prefix": "AP: "},
            steps=steps,
        )
    ],
    title=f"Allen CCF coronal slice, AP = {coords_mm[0]:.2f} mm",
    width=700,
    height=700,
    margin=dict(l=0, r=0, t=50, b=40),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

fig.update_geos(
    visible=False,
    fitbounds="locations",
    projection_type="equirectangular",
)

fig.show()